# Data Pipeline and Model Sanity Check
quick verification that the dataset, model, and training pipeline all work

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
from src.data.salicon_coco import SaliconCocoDataset
from src.data.transforms import build_image_transform, SaliencyTransform
from src.config import SALICON_DIR, COCO_ANN_DIR, NUM_LABELS
from src.utils.viz import show_sample

In [ ]:
# load dataset
img_tfm = build_image_transform(train=False)
sal_tfm = SaliencyTransform()

train_ds = SaliconCocoDataset(
    img_dir=SALICON_DIR / 'images' / 'train',
    saliency_dir=SALICON_DIR / 'train',
    coco_ann_path=COCO_ANN_DIR / 'instances_train2014.json',
    split_prefix='COCO_train2014',
    img_tfm=img_tfm, sal_tfm=sal_tfm, k_labels=NUM_LABELS,
)
print(f'train samples: {len(train_ds)}')
print(f'categories: {train_ds.cat_names}')

In [ ]:
# show a few samples
for i in [0, 42, 100, 500]:
    sample = train_ds[i]
    show_sample(sample['image'], sample['sal_full'], sample['sal_grid'],
               y_cls=sample['y_cls'], cat_names=train_ds.cat_names)

In [ ]:
# load trained model and run inference
from src.models.vit_multitask import build_model
from src.utils.io import load_checkpoint
from src.config import RUNS_DIR, DEVICE

model, _ = build_model(mode='multitask', num_labels=NUM_LABELS, pretrained=False)
load_checkpoint(model, RUNS_DIR / 'best_multitask.pt')
model = model.to(DEVICE)
model.eval()

# run on a sample
sample = train_ds[42]
x = sample['image'].unsqueeze(0).to(DEVICE)
with torch.no_grad():
    logits, sal_pred = model(x)
    probs = torch.sigmoid(logits[0])

# show predictions
print('predicted categories:')
for i in range(NUM_LABELS):
    if probs[i] > 0.5:
        print(f'  {train_ds.cat_names[i]}: {probs[i]:.3f}')
print(f'\nground truth: {[train_ds.cat_names[i] for i in range(NUM_LABELS) if sample["y_cls"][i] > 0.5]}')

In [ ]:
# compare predicted saliency to ground truth
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
from src.utils.viz import denormalize_image

axes[0].imshow(denormalize_image(sample['image']))
axes[0].set_title('image')
axes[0].axis('off')

axes[1].imshow(sample['sal_grid'].squeeze().numpy(), cmap='hot')
axes[1].set_title('ground truth saliency')
axes[1].axis('off')

axes[2].imshow(sal_pred[0].squeeze().cpu().numpy(), cmap='hot')
axes[2].set_title('predicted saliency')
axes[2].axis('off')

plt.tight_layout()
plt.show()